# Hybrid Experiments: Latest XGB + Fusion

### Imports

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import xgboost as xgb
import numpy as np
import librosa as lb
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras import models, layers, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

## Branch 1: **Engineered XGBoost Features**

Dataset now features:

- 12 kHz target sample rate,
- 6-second chunks,
- overlap for non-COPD classes using a 2-second step,
- no overlap for COPD,
- Mu-law compression,
- and a richer feature set including stds, bandwidth, skewness/kurtosis, and MFCCs up to 15.

### Load XGB data

In [2]:
xgb_df = pd.read_csv("/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Extracted features/xgboost_features_6s_12kHz_compressed_after_normalization_w_overlapping_mfcc15.csv")

xgb_df.head()

,rms_mean,rms_std,zcr_mean,centroid_mean,centroid_std,flatness_mean,flatness_std,rolloff_mean,flux_mean,bandwidth_mean,...,mfcc_13_mean,mfcc_13_std,mfcc_14_mean,mfcc_14_std,mfcc_15_mean,mfcc_15_std,patient_id,chunk_id,original_file,diagnosis
0,0.740132,0.083106,0.001316,98.066230,93.868556,0.000034,0.000146,119.431516,1.826715,340.148743,...,8.284259,2.947182,8.153688,3.566122,7.156815,3.025179,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.696974,0.079933,0.001572,94.479942,81.417895,0.000041,0.000361,107.338763,1.494651,338.655438,...,8.452517,3.363525,7.557367,4.090311,6.353885,4.028513,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.670958,0.079359,0.001590,90.514388,75.801911,0.000024,0.000205,99.817154,1.613105,339.537651,...,8.950355,2.893862,7.780566,3.040372,6.503709,3.147859,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.675636,0.093185,0.001153,81.297469,52.764923,0.000008,0.000023,81.740359,1.693879,322.028853,...,8.722843,2.706289,8.697692,3.737317,6.902027,3.432181,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.635533,0.069224,0.001202,80.768717,67.285689,0.000014,0.000083,81.865027,1.705645,325.092538,...,9.458111,3.485257,10.628664,4.948114,8.951859,4.647463,223,4,223_1b1_Pr_sc_Meditron.wav,COPD


In [3]:
xgb_df.shape, xgb_df.columns, xgb_df.dtypes

((3606, 46),
 Index(['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std',
        'flatness_mean', 'flatness_std', 'rolloff_mean', 'flux_mean',
        'bandwidth_mean', 'skewness_mean', 'kurtosis_mean', 'mfcc_1_mean',
        'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std',
        'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean',
        'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std',
        'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std',
        'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std',
        'mfcc_13_mean', 'mfcc_13_std', 'mfcc_14_mean', 'mfcc_14_std',
        'mfcc_15_mean', 'mfcc_15_std', 'patient_id', 'chunk_id',
        'original_file', 'diagnosis'],
       dtype='object'),
 rms_mean          float64
 rms_std           float64
 zcr_mean          float64
 centroid_mean     float64
 centroid_std      float64
 flatness_mean     float64
 flatness_std      float64


### XGB preprocessing

#### Apply 5-class filter

In [4]:
# Filter to same 5 classes - COPD, pneumonia, healthy, URTI, bronchiectasis
classes_to_keep = ["COPD", "Pneumonia", "Healthy", "URTI", "Bronchiectasis"]

xgb_df_filtered = xgb_df[xgb_df["diagnosis"].isin(classes_to_keep)].copy()
xgb_df_filtered = xgb_df_filtered.reset_index(drop=True)

In [5]:
xgb_df_filtered["diagnosis"].value_counts()

diagnosis
COPD              2597
Pneumonia          296
Healthy            275
URTI               182
Bronchiectasis     128
Name: count, dtype: int64

#### Label-encode target

In [6]:
# Encode target
le = LabelEncoder()
xgb_df_filtered["target"] = le.fit_transform(xgb_df_filtered["diagnosis"])

# Create a dictionary mapping labels -> numbers
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

class_mapping

{'Bronchiectasis': 0, 'COPD': 1, 'Healthy': 2, 'Pneumonia': 3, 'URTI': 4}

### Train & Test with Grouped Cross Validation

In [7]:
# Define features and target
X = xgb_df_filtered.drop(columns=["original_file", "patient_id", "diagnosis", "target", "chunk_id"])
y = xgb_df_filtered["target"]

# Define groups (by patient)
groups = xgb_df_filtered["patient_id"]

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of groups (patients): {groups.nunique()}")
print("\nConfirm feature columns:")
print(list(X.columns))


Feature matrix shape: (3478, 42)
Target shape: (3478,)
Number of groups (patients): 117

Confirm feature columns:
['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std', 'flatness_mean', 'flatness_std', 'rolloff_mean', 'flux_mean', 'bandwidth_mean', 'skewness_mean', 'kurtosis_mean', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std', 'mfcc_14_mean', 'mfcc_14_std', 'mfcc_15_mean', 'mfcc_15_std']


In [8]:
# Grouped Cross Validation (split by patient ID)
# IMPORTANT: We used StratifiedGroupKFold to preserve patient-level separation
# while improving class balance across folds.
sgkf = StratifiedGroupKFold(n_splits=3)

fold_results = []

# Store probabilities across all folds
# (for use in fusion model)
xgb_val_proba_all = []
xgb_test_proba_all = []
y_val_all = []
y_test_all = []

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    # Outer split: creates grouped train/test splits
    X_train_full = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_full = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train_full = groups.iloc[train_idx]

    # Inner split: creates grouped train/validation splits
    gss_val = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
    idx_train, idx_val = next(gss_val.split(X_train_full, y_train_full, groups=groups_train_full))

    X_train = X_train_full.iloc[idx_train]
    X_val = X_train_full.iloc[idx_val]

    y_train = y_train_full.iloc[idx_train]
    y_val = y_train_full.iloc[idx_val]

    # Class-balance sample weights on TRAIN only
    w_train = compute_sample_weight(class_weight="balanced", y=y_train)

    # ==================
    # XGBoost model
    # ==================
    # UPDATED to match latest model from Antonella
    xgb_model = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=3,
        subsample=0.8,        # only 80% of lines in each tree
        colsample_bytree=0.7, # only 70% of cols in each tree
        reg_lambda=1.5,
        objective='multi:softprob',
        num_class=len(le.classes_),
        random_state=42,
        eval_metric='mlogloss',
        early_stopping_rounds=15
    )

    print("Train classes:", np.sort(y_train.unique()))
    print("Val classes:  ", np.sort(y_val.unique()))
    print("Test classes: ", np.sort(y_test.unique()))

    # Train
    xgb_model.fit(
        X_train,
        y_train,
        sample_weight=w_train, # gives more importance to rare classes
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # Predict class labels on test set
    y_pred = xgb_model.predict(X_test)

    # NEW: predict probabilities on validation + test sets
    xgb_val_proba = xgb_model.predict_proba(X_val)
    xgb_test_proba = xgb_model.predict_proba(X_test)

    # NEW: store validation + test probabilities and matching labels
    xgb_val_proba_all.append(xgb_val_proba)
    xgb_test_proba_all.append(xgb_test_proba)
    y_val_all.append(y_val.values)
    y_test_all.append(y_test.values)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Fold {fold} accuracy   : {acc:.4f}")
    print(f"Fold {fold} macro F1   : {macro_f1:.4f}")
    print(f"Fold {fold} weighted F1: {weighted_f1:.4f}\n")

    print(classification_report(
        y_test,
        y_pred,
        target_names=le.classes_,
        zero_division=0
    ))

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

# =========================================================
# Summary across folds
# =========================================================
results_df = pd.DataFrame(fold_results)

print("\n" + "="*60)
print("CROSS-VALIDATION SUMMARY")
print("="*60)
print(results_df)

print("\nMean metrics:")
print(results_df[["accuracy", "macro_f1", "weighted_f1"]].mean())

# NEW: combine probabilities + labels from all folds
xgb_val_proba_all = np.vstack(xgb_val_proba_all)
xgb_test_proba_all = np.vstack(xgb_test_proba_all)
y_val_all = np.concatenate(y_val_all)
y_test_all = np.concatenate(y_test_all)

print("\nValidation probability shape across all folds:")
print(xgb_val_proba_all.shape)

print("Validation labels shape across all folds:")
print(y_val_all.shape)

print("\nTest probability shape across all folds:")
print(xgb_test_proba_all.shape)

print("Test labels shape across all folds:")
print(y_test_all.shape)


FOLD 1
Train classes: [0 1 2 3 4]
Val classes:   [1 2 3 4]
Test classes:  [0 1 2 3 4]
Fold 1 accuracy   : 0.7185
Fold 1 macro F1   : 0.4369
Fold 1 weighted F1: 0.7377

                precision    recall  f1-score   support

Bronchiectasis       0.58      0.62      0.60        40
          COPD       0.94      0.84      0.89       865
       Healthy       0.27      0.51      0.35        94
     Pneumonia       0.25      0.27      0.26        96
          URTI       0.09      0.08      0.09        63

      accuracy                           0.72      1158
     macro avg       0.43      0.47      0.44      1158
  weighted avg       0.77      0.72      0.74      1158


FOLD 2
Train classes: [0 1 2 3 4]
Val classes:   [0 1 2 3 4]
Test classes:  [0 1 2 3 4]
Fold 2 accuracy   : 0.7961
Fold 2 macro F1   : 0.4786
Fold 2 weighted F1: 0.7832

                precision    recall  f1-score   support

Bronchiectasis       0.52      0.40      0.45        40
          COPD       0.90      0.95     

## Branch 2: **CNN**

Adapt config to latest XGBoost set-up

In [9]:
cnn_df = xgb_df_filtered.copy()

cnn_df.shape, cnn_df["diagnosis"].value_counts()

((3478, 47),
 diagnosis
 COPD              2597
 Pneumonia          296
 Healthy            275
 URTI               182
 Bronchiectasis     128
 Name: count, dtype: int64)

### Set-up with new chunk logic

In [10]:
# Match XGBoost preprocessing
# - UPDATED: 12 kHz sample rate
# - fixed 6-second slices
TARGET_SR = 12000
CHUNK_DURATION = 6
SAMPLES_PER_CHUNK = TARGET_SR * CHUNK_DURATION

RAW_AUDIO_FOLDER = Path("/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files")

# NEW Overlap logic
STEP_COPD = SAMPLES_PER_CHUNK          # 6s step
STEP_NON_COPD = TARGET_SR * 2          # 2s step

In [11]:
def load_one_chunk_with_overlap(original_file, chunk_id, diagnosis, raw_audio_folder):
    """
    Reconstructs the exact chunk using Antonella's logic:
    - COPD → no overlap (6s step)
    - Others → 2s step (overlapping)
    """

    # Load audio at 12kHz
    y, _ = lb.load(raw_audio_folder / original_file, sr=TARGET_SR)

    # Trim silence (same as before)
    y, _ = lb.effects.trim(y)

    # Choose step size based on class
    if diagnosis == "COPD":
        step = STEP_COPD
    else:
        step = STEP_NON_COPD

    # Create chunks using correct step
    chunks = [
        y[i:i + SAMPLES_PER_CHUNK]
        for i in range(0, len(y), step)
        if len(y[i:i + SAMPLES_PER_CHUNK]) == SAMPLES_PER_CHUNK
    ]

    # Return correct chunk
    return chunks[int(chunk_id)].astype(np.float32)

### Create MelSpec from new chunks

In [12]:
def chunk_to_mel(chunk):
    mel = lb.feature.melspectrogram(
        y=chunk,
        sr=TARGET_SR,
        n_mels=128
    )

    mel_db = lb.power_to_db(mel, ref=np.max)

    # Add channel dimension for CNN
    return mel_db[..., np.newaxis].astype(np.float32)

In [13]:
def build_cnn_dataset(df_subset, raw_audio_folder):
    """
    Builds mel spectrogram dataset aligned with new chunking logic
    """

    X_list = []
    y_list = []

    for row in df_subset.itertuples(index=False):
        chunk = load_one_chunk_with_overlap(
            row.original_file,
            row.chunk_id,
            row.diagnosis,
            raw_audio_folder
        )

        mel = chunk_to_mel(chunk)

        X_list.append(mel)
        y_list.append(row.target)

    return np.stack(X_list), np.array(y_list)

### Reuse existing splits for CNN

In [14]:
# Grouped split (using same logics as XGBoost)
X_dummy = cnn_df.drop(columns=["diagnosis", "target"])
y = cnn_df["target"]
groups = cnn_df["patient_id"]

# Use same indices as above
X_train = X_train_full.iloc[idx_train]
X_val = X_train_full.iloc[idx_val]
X_test = X.iloc[test_idx]

# ==================
# Align CNN splits with XGB
# ==================
cnn_df_train_full = cnn_df.iloc[train_idx]
cnn_df_test = cnn_df.iloc[test_idx]

cnn_df_train = cnn_df_train_full.iloc[idx_train]
cnn_df_val = cnn_df_train_full.iloc[idx_val]

In [15]:
# Build mel datasets
X_train_img, y_train = build_cnn_dataset(cnn_df_train, RAW_AUDIO_FOLDER)
X_val_img, y_val = build_cnn_dataset(cnn_df_val, RAW_AUDIO_FOLDER)
X_test_img, y_test = build_cnn_dataset(cnn_df_test, RAW_AUDIO_FOLDER)

In [16]:
print(X_train_img.shape)
print(y_train.shape)

(1943, 128, 141, 1)
(1943,)


In [17]:
# Checking class representation
print(np.unique(y_train))

[0 1 2 3 4]


### Train and evaluate CNN

In [18]:
# One-hot encode labels
num_classes = len(np.unique(y_train))

y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [19]:
def build_cnn(input_shape, num_classes):
    cnn_model = models.Sequential()

    #Input
    cnn_model.add(layers.Input(shape=input_shape))

    # Conv2D Block 1
    cnn_model.add(layers.Conv2D(32, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Conv2D Block 2
    cnn_model.add(layers.Conv2D(64, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Conv2D Block 3
    cnn_model.add(layers.Conv2D(128, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Conv2D Block 4
    cnn_model.add(layers.Conv2D(256, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Turn feature maps into one vector
    cnn_model.add(layers.GlobalMaxPooling2D())

    # Dense layer before classification
    cnn_model.add(layers.Dense(32, activation="relu"))
    cnn_model.add(layers.Dropout(0.3))

    # Final prediction layer
    cnn_model.add(layers.Dense(num_classes, activation="softmax"))

    cnn_model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return cnn_model

cnn_model = build_cnn(X_train_img.shape[1:], num_classes)
cnn_model.summary()

2026-03-24 11:36:26.390180: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-24 11:36:26.390297: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-24 11:36:26.390315: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-03-24 11:36:26.390495: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-24 11:36:26.390512: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 141, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 141, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 141, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 70, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 70, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 70, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 70, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 35, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 35, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 35, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 35, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 17, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 17, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 17, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 256)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 398,149 (1.52 MB)

 Trainable params: 397,189 (1.52 MB)

 Non-trainable params: 960 (3.75 KB)

In [20]:
cnn_model.fit(
    X_train_img,
    y_train_cat,
    validation_data=(X_val_img, y_val_cat),
    epochs=30,
    batch_size=32,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

Epoch 1/30


2026-03-24 11:36:27.524506: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


61/61 ━━━━━━━━━━━━━━━━━━━━ 9s 111ms/step - accuracy: 0.5991 - loss: 2.5134 - val_accuracy: 0.7513 - val_loss: 1.9776
Epoch 2/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 6s 98ms/step - accuracy: 0.6423 - loss: 1.8765 - val_accuracy: 0.7513 - val_loss: 1.3726
Epoch 3/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.7030 - loss: 1.3832 - val_accuracy: 0.7513 - val_loss: 2.3470
Epoch 4/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.7494 - loss: 1.0951 - val_accuracy: 0.8063 - val_loss: 0.8956
Epoch 5/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.7787 - loss: 1.0126 - val_accuracy: 0.8141 - val_loss: 0.8032
Epoch 6/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.7941 - loss: 1.0131 - val_accuracy: 0.7513 - val_loss: 5.6839
Epoch 7/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 6s 96ms/step - accuracy: 0.7905 - loss: 0.9764 - val_accuracy: 0.6990 - val_loss: 1.4741
Epoch 8/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.8060 - loss: 0.9311 - val_accuracy: 0.7513 - val_loss: 3

In [21]:
cnn_proba = cnn_model.predict(X_test_img)

cnn_proba

37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step


array([[1.4829754e-09, 9.9981755e-01, 1.7270471e-04, 1.0230228e-06,
        8.6421051e-06],
       [2.5320237e-09, 9.9998629e-01, 1.0481979e-05, 1.2989528e-07,
        3.1464444e-06],
       [9.7537436e-09, 9.9996483e-01, 2.9478613e-05, 3.1437480e-07,
        5.3555682e-06],
       ...,
       [8.0967531e-08, 9.9998879e-01, 1.0842747e-05, 1.4873259e-09,
        3.2476737e-07],
       [8.4477995e-07, 9.9996209e-01, 3.4821562e-05, 2.2833280e-08,
        2.2924289e-06],
       [2.3371763e-07, 9.9999952e-01, 2.2902297e-07, 4.6673898e-10,
        9.4619175e-09]], dtype=float32)

In [22]:
cnn_proba.shape

(1153, 5)

In [23]:
cnn_val_proba = cnn_model.predict(X_val_img)
cnn_test_proba = cnn_model.predict(X_test_img)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step


## Fusion: XGB and CNN

In [24]:
# =========================================================
# Grouped Cross Validation (shared for XGB + CNN)
# =========================================================

sgkf = StratifiedGroupKFold(n_splits=3)

fold_results = []

# Store across folds
xgb_val_proba_all = []
xgb_test_proba_all = []
cnn_val_proba_all = []
cnn_test_proba_all = []

y_val_all = []
y_test_all = []

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):

    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    # =========================================================
    # Outer split
    # =========================================================
    X_train_full = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_full = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train_full = groups.iloc[train_idx]

    # =========================================================
    # Inner split (train / val)
    # =========================================================
    gss_val = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)

    idx_train, idx_val = next(
        gss_val.split(X_train_full, y_train_full, groups=groups_train_full)
    )

    X_train = X_train_full.iloc[idx_train]
    X_val = X_train_full.iloc[idx_val]

    y_train = y_train_full.iloc[idx_train]
    y_val = y_train_full.iloc[idx_val]

    print("Train classes:", np.sort(y_train.unique()))
    print("Val classes:  ", np.sort(y_val.unique()))
    print("Test classes: ", np.sort(y_test.unique()))

    # =========================================================
    # XGBoost
    # =========================================================
    w_train = compute_sample_weight(class_weight="balanced", y=y_train)

    xgb_model = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=1.5,
        objective='multi:softprob',
        num_class=len(le.classes_),
        random_state=42,
        eval_metric='mlogloss',
        early_stopping_rounds=15
    )

    xgb_model.fit(
        X_train,
        y_train,
        sample_weight=w_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # XGB predictions
    xgb_val_proba = xgb_model.predict_proba(X_val)
    xgb_test_proba = xgb_model.predict_proba(X_test)

    # =========================================================
    # CNN (using SAME indices)
    # =========================================================
    cnn_df_train_full = cnn_df.iloc[train_idx]
    cnn_df_test = cnn_df.iloc[test_idx]

    cnn_df_train = cnn_df_train_full.iloc[idx_train]
    cnn_df_val = cnn_df_train_full.iloc[idx_val]

    # Build datasets
    X_train_img, y_train_img = build_cnn_dataset(cnn_df_train, RAW_AUDIO_FOLDER)
    X_val_img, y_val_img = build_cnn_dataset(cnn_df_val, RAW_AUDIO_FOLDER)
    X_test_img, y_test_img = build_cnn_dataset(cnn_df_test, RAW_AUDIO_FOLDER)

    # One-hot encode
    num_classes = len(le.classes_)
    y_train_cat = to_categorical(y_train_img, num_classes)
    y_val_cat = to_categorical(y_val_img, num_classes)

    # Build CNN
    cnn_model = build_cnn(X_train_img.shape[1:], num_classes)

    cnn_model.fit(
        X_train_img,
        y_train_cat,
        validation_data=(X_val_img, y_val_cat),
        epochs=30,
        batch_size=32,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
        verbose=0
    )

    # CNN predictions
    cnn_val_proba = cnn_model.predict(X_val_img, verbose=0)
    cnn_test_proba = cnn_model.predict(X_test_img, verbose=0)

    # =========================================================
    # Store everything
    # =========================================================
    xgb_val_proba_all.append(xgb_val_proba)
    xgb_test_proba_all.append(xgb_test_proba)

    cnn_val_proba_all.append(cnn_val_proba)
    cnn_test_proba_all.append(cnn_test_proba)

    y_val_all.append(y_val.values)
    y_test_all.append(y_test.values)

    # =========================================================
    # Metrics (XGB only for baseline tracking)
    # =========================================================
    y_pred = np.argmax(xgb_test_proba, axis=1)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Fold {fold} accuracy   : {acc:.4f}")
    print(f"Fold {fold} macro F1   : {macro_f1:.4f}")
    print(f"Fold {fold} weighted F1: {weighted_f1:.4f}\n")

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

# =========================================================
# Combine everything
# =========================================================
xgb_val_proba_all = np.vstack(xgb_val_proba_all)
xgb_test_proba_all = np.vstack(xgb_test_proba_all)

cnn_val_proba_all = np.vstack(cnn_val_proba_all)
cnn_test_proba_all = np.vstack(cnn_test_proba_all)

y_val_all = np.concatenate(y_val_all)
y_test_all = np.concatenate(y_test_all)

print("\nFinal shapes:")
print("xgb_val:", xgb_val_proba_all.shape)
print("cnn_val:", cnn_val_proba_all.shape)
print("xgb_test:", xgb_test_proba_all.shape)
print("cnn_test:", cnn_test_proba_all.shape)
print("y_test:", y_test_all.shape)


FOLD 1
Train classes: [0 1 2 3 4]
Val classes:   [1 2 3 4]
Test classes:  [0 1 2 3 4]
Fold 1 accuracy   : 0.7185
Fold 1 macro F1   : 0.4369
Fold 1 weighted F1: 0.7377


FOLD 2
Train classes: [0 1 2 3 4]
Val classes:   [0 1 2 3 4]
Test classes:  [0 1 2 3 4]
Fold 2 accuracy   : 0.7961
Fold 2 macro F1   : 0.4786
Fold 2 weighted F1: 0.7832


FOLD 3
Train classes: [0 1 2 3 4]
Val classes:   [0 1 2 4]
Test classes:  [0 1 2 3 4]
Fold 3 accuracy   : 0.8265
Fold 3 macro F1   : 0.5138
Fold 3 weighted F1: 0.8027


Final shapes:
xgb_val: (1621, 5)
cnn_val: (1621, 5)
xgb_test: (3478, 5)
cnn_test: (3478, 5)
y_test: (3478,)


In [25]:
# Find optimum weighting

weights = np.linspace(0, 1, 11)  # finer search: 0.0 → 1.0

best_w = None
best_score = -1

for w in weights:
    final_proba = w * xgb_val_proba_all + (1 - w) * cnn_val_proba_all
    y_pred = np.argmax(final_proba, axis=1)

    score = f1_score(y_val_all, y_pred, average="macro")

    if score > best_score:
        best_score = score
        best_w = w

    print(f"w={w:.2f} → Macro F1: {score:.4f}")

print("\nBest weight:", best_w)
print("Best val macro F1:", best_score)

w=0.00 → Macro F1: 0.4803
w=0.10 → Macro F1: 0.4833
w=0.20 → Macro F1: 0.4916
w=0.30 → Macro F1: 0.4868
w=0.40 → Macro F1: 0.5040
w=0.50 → Macro F1: 0.5429
w=0.60 → Macro F1: 0.6166
w=0.70 → Macro F1: 0.6206
w=0.80 → Macro F1: 0.6274
w=0.90 → Macro F1: 0.6152
w=1.00 → Macro F1: 0.5990

Best weight: 0.8
Best val macro F1: 0.6273678508108572


In [27]:
best_w = best_w  # from tuning

# Fuse test probabilities
final_test_proba = best_w * xgb_test_proba_all + (1 - best_w) * cnn_test_proba_all

# Convert probabilities to predicted classes
y_test_pred = np.argmax(final_test_proba, axis=1)

# Evaluate
print("\nHYBRID RESULTS")
print("Macro F1:", f1_score(y_test_all, y_test_pred, average="macro"))
print("Weighted F1:", f1_score(y_test_all, y_test_pred, average="weighted"))
print("Accuracy:", accuracy_score(y_test_all, y_test_pred))

print(classification_report(
    y_test_all,
    y_test_pred,
    target_names=le.classes_,
    zero_division=0
))


HYBRID RESULTS
Macro F1: 0.4988696401568239
Weighted F1: 0.7876540509096791
Accuracy: 0.8013225991949396
                precision    recall  f1-score   support

Bronchiectasis       0.69      0.44      0.54       128
          COPD       0.90      0.94      0.92      2597
       Healthy       0.46      0.61      0.52       275
     Pneumonia       0.51      0.32      0.39       296
          URTI       0.16      0.10      0.12       182

      accuracy                           0.80      3478
     macro avg       0.54      0.48      0.50      3478
  weighted avg       0.78      0.80      0.79      3478



#### What changed for minority classes?

In [28]:
y_xgb_pred = np.argmax(xgb_test_proba_all, axis=1)
y_hybrid_pred = np.argmax(final_test_proba, axis=1)

In [29]:
from sklearn.metrics import classification_report

xgb_report = classification_report(
    y_test_all,
    y_xgb_pred,
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

hybrid_report = classification_report(
    y_test_all,
    y_hybrid_pred,
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

In [30]:
import pandas as pd

rows = []

for cls in le.classes_:
    rows.append({
        "class": cls,
        "xgb_f1": xgb_report[cls]["f1-score"],
        "hybrid_f1": hybrid_report[cls]["f1-score"],
        "change": hybrid_report[cls]["f1-score"] - xgb_report[cls]["f1-score"]
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df)

            class    xgb_f1  hybrid_f1    change
0  Bronchiectasis  0.575107   0.535885 -0.039222
1            COPD  0.915177   0.919670  0.004493
2         Healthy  0.433180   0.523364  0.090185
3       Pneumonia  0.344969   0.392562  0.047593
4            URTI  0.097859   0.122867  0.025008
